# 01 — Resume PDF Extraction
Member 1 data pipeline: raw PDF resumes → canonical `data/processed/resumes.csv`.

Run notebooks from the repository root (`notebooks/` is auto-detected).

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pymupdf
print('Root:', ROOT)

Root: C:\Users\JIYA SADARIA\OneDrive\Pictures\Desktop\samatrix\SAMATRIX-RESUMEFORGE-_vision-x


## OCR decision check
Sample 20 PDFs and measure how many contain extractable text.

In [2]:
import random
DATA = ROOT / 'data' / 'raw' / 'resume_dataset' / 'data' / 'data'
pdfs = sorted(DATA.rglob('*.pdf'), key=lambda p: (p.parent.name, p.name))
print('Total PDFs:', len(pdfs))
random.seed(42)
sample = random.sample(pdfs, 20)
ok = 0
for p in sample:
    with pymupdf.open(str(p)) as doc:
        t = ''.join(pg.get_text() for pg in doc)
    if len(t.strip()) > 50:
        ok += 1
print(f'{ok}/20 sampled PDFs had extractable text.')
print('Decision: normal text extraction is sufficient — no OCR pipeline needed.')

Total PDFs: 2500


20/20 sampled PDFs had extractable text.
Decision: normal text extraction is sufficient — no OCR pipeline needed.


## Run extraction
Produces `data/processed/resumes.csv`, `reports/eda/extraction_report.json`, and failures CSV when needed.

In [3]:
import subprocess
result = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'data' / 'extract_resumes.py')], cwd=ROOT)
print('exit code:', result.returncode)

exit code: 0


## Inspect the canonical dataset

In [4]:
import pandas as pd, json
df = pd.read_csv(ROOT / 'data' / 'processed' / 'resumes.csv')
print(df.shape)
print(df.head(3)[['resume_id','filename','category']])
print(df['category'].nunique(), 'categories')
print(json.loads((ROOT / 'reports' / 'eda' / 'extraction_report.json').read_text()))

(2500, 4)
   resume_id      filename    category
0  RF-000001  10554236.pdf  ACCOUNTANT
1  RF-000002  10674770.pdf  ACCOUNTANT
2  RF-000003  11163645.pdf  ACCOUNTANT
24 categories
{'total_files': 2500, 'successful_extractions': 2500, 'failed_extractions': 0, 'empty_extractions': 1, 'categories': ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER'], 'category_counts': {'ACCOUNTANT': 118, 'ADVOCATE': 118, 'AGRICULTURE': 63, 'APPAREL': 97, 'ARTS': 103, 'AUTOMOBILE': 36, 'AVIATION': 117, 'BANKING': 115, 'BPO': 22, 'BUSINESS-DEVELOPMENT': 120, 'CHEF': 118, 'CONSTRUCTION': 112, 'CONSULTANT': 115, 'DESIGNER': 107, 'DIGITAL-MEDIA': 96, 'ENGINEERING': 118, 'FINANCE': 134, 'FITNESS': 117, 'HEALTHCARE': 115, 'HR': 110, 'INFORMATION-TEC